I will be predicting whether the 'Man of the match' winner came from a given team after a match has ended. A match contributes two teams rows, one labeled 1 and the other 0. The two rows are linked observation, so later splitting and evaluation happen at the match level.

In [1]:
import numpy as np
import pandas as pd

# Relative path: the CSV sits next to the notebook, so this runs on any machine.
DATASET_PATH = "world_cup_team_statistics.csv"
TARGET = "Man of the Match"

df = pd.read_csv(DATASET_PATH)

print(df.shape) #show rows and columns

print(df["match_id"].nunique()) #show number of matches

print(df["year"].value_counts().sort_index()) #tournament years and number of matches played in each year

print("Target counts:")
print(df[TARGET].value_counts().sort_index())

print(df.head())

(256, 45)
128
year
2018    128
2022    128
Name: count, dtype: int64
Target counts:
Man of the Match
0    128
1    128
Name: count, dtype: int64
         Date          Team      Opponent  Goal Scored  Ball Possession %  \
0  2018-06-14        Russia  Saudi Arabia            5               40.0   
1  2018-06-14  Saudi Arabia        Russia            0               60.0   
2  2018-06-15         Egypt       Uruguay            0               43.0   
3  2018-06-15       Uruguay         Egypt            1               57.0   
4  2018-06-15          Iran       Morocco            1               36.0   

   Attempts  On-Target  Off-Target  Blocked  Corners  ...  \
0        13          7           3        3        6  ...   
1         6          0           3        3        2  ...   
2         8          3           3        2        0  ...   
3        14          4           6        4        5  ...   
4         8          2           5        1        2  ...   

   award_winner_player_id

The target is balanced by construction, but the effective sample size is 128 matches rather than 256 independent decisions. passes_completed, possession_in_contest_pct, incomplete_attempts, and other supplemental columns are available only for 2022. The 2022 statistics_method says possession was normalized to exclude an in-contest share and pass accuracy was computed.
I will use shared match-stat columns for the main comparison and report year-specific results later.

In [2]:
assert df["match_id"].notna().all() # check if there are any missing values in the match_id column
assert df["Team"].notna().all() # check if there are any missing values in the Team column
assert df["Opponent"].notna().all() # check if there are any missing values in the Opponent column
assert df[TARGET].notna().all() # check if there are any missing values in the Man of the Match column
assert set(df[TARGET].unique()) == {0, 1} # check if the Man of the Match column only contains 0 and 1

match_check = df.groupby("match_id").agg(
    rows=(TARGET, "size"), 
    positive_labels=(TARGET, "sum"),
    years=("year", "nunique"),
) 

assert (match_check["rows"]  == 2).all() #check if there are exactly 2 rows for each match_id
assert (match_check["positive_labels"]  == 1).all() #check if there is exactly 1 positive label for each match_id
assert (match_check["years"]  == 1).all() #check if there is exactly 1 unique year for each match_id
assert not df.duplicated(subset=["match_id", "Team"]).any() #check if there are any duplicate rows for the same match_id and Team combination

for _, pair in df.groupby("match_id"):
    a, b = pair.iloc[0], pair.iloc[1]
    assert a["Team"] == b["Opponent"]
    assert b["Team"] == a["Opponent"]

print("All matches have two reciprocal teams and one positive label for Man of the Match.")

All matches have two reciprocal teams and one positive label for Man of the Match.


Preprocessing for inspecting missingness and source differences

In [3]:
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_rows"))

source_columns = [
    "award_status", "statistics_source", "award_source", "statistics_method"
]
for column in source_columns:
    print(f"\n{column}")
    display(pd.crosstab(df["year"], df[column]))

display(
    df.groupby("year")[[
        "Ball Possession %", "Pass Accuracy %", "Passes",
        "Distance Covered (Kms)", "Attempts"
    ]].agg(["mean", "min", "max"]).round(2)
)


,missing_rows
Own goal Time,242
award_winner_player,128
passes_completed,128
on_target_shot_outcomes,128
on_target_fifa_summary,128
incomplete_attempts,128
possession_in_contest_pct,128
fifa_report_url,128
award_winner_player_id,128
1st Goal,73



award_status


award_status,reported,reported_online
year,,
2018,128,0
2022,0,128



statistics_source


statistics_source,FIFA 2018 Statistics.csv,"FIFA post-match report; supplemental match CSV for corners, offsides and fouls"
year,,
2018,128,0
2022,0,128



award_source


award_source,FIFA 2018 Statistics.csv (user supplied),https://www.soccer-blogger.com/2022/12/17/2022-football-world-cup-man-of-the-match-winners-list-of-motm-awards-at-fifa-wc-2022/
year,,
2018,128,0
2022,0,128



statistics_method


statistics_method,FIFA normalized possession excluding in-contest share; computed pass accuracy,Supplied 2018 statistics
year,,
2018,0,128
2022,128,0


Ball Possession %               Pass Accuracy %                Passes  \
                  mean    min    max            mean    min    max    mean   
year                                                                         
2018              50.0  25.00  75.00           82.55  67.00  94.00  462.65   
2022              50.0  15.82  84.18           84.11  65.38  94.53  500.80   

                Distance Covered (Kms)               Attempts          
      min   max                   mean    min    max     mean min max  
year                                                                   
2018  189  1137                 106.66   80.0  148.0    12.59   3  26  
2022  224  1070                 115.25  102.3  148.2    11.21   0  32

The target is balanced, I will use shared match-stat columns for the main comparison and report year-specific results later.

## Phase 2: features and split

**Feature policy.** I use the 19 numeric stats that both 2018 and 2022 record. I leave out the award columns (they give the answer away), identifiers, source metadata and 2022-only columns. `Man of the Match` is never an input: it is the answer key during training and the grading key after testing.

Goals nearly reveal the answer, so I build two feature sets:
- **with goals:** all 19 stats.
- **no goals:** drops `Goal Scored`, `Goals in PSO` and `Own goals`, the columns that give the score away. This is my **main** model. It won't be more accurate (goals make the task easier), but it tests whether *how a team played* predicts the award.

There is no minutes-played column, so extra-time matches keep their inflated raw counts (passes, shots, distance). I keep those matches and note this as a limitation.

In [4]:
# The logic lives in motm.py (with tests in test_motm.py) so these cells stay short.
import motm
from motm import FEATURE_SETS, FEATURES_WITH_GOALS, FEATURES_NO_GOALS

assert TARGET == motm.TARGET
for name, features in FEATURE_SETS.items():
    assert TARGET not in features
    assert len(features) == len(set(features))
    assert set(features).issubset(df.columns)
    print(f"{name}: {len(features)} features")

print("Dropped for the no-goals set:", motm.SCORE_LEAK_COLUMNS)

# Turn any stray text into NaN. The imputer inside each pipeline fills gaps
# using training rows only.
df = motm.coerce_features(df, FEATURES_WITH_GOALS)
display(df[list(FEATURES_WITH_GOALS)].isna().sum().to_frame("missing_after_conversion"))
assert not df[list(FEATURES_WITH_GOALS)].isna().all().any()

with goals: 19 features
no goals: 16 features
Dropped for the no-goals set: ('Goal Scored', 'Goals in PSO', 'Own goals')


,missing_after_conversion
Goal Scored,0
Ball Possession %,0
Attempts,0
On-Target,0
Off-Target,0
Blocked,0
Corners,0
Offsides,0
Free Kicks,0
Saves,0


In [5]:
# Split whole matches 80/20, stratified by year. Both rows of a match land on
# the same side, so the model never sees one team in training and its mirrored
# opponent in testing.
split = motm.split_by_match(df)
train_df, test_df = split.train, split.test
y_train, y_test = train_df[TARGET], test_df[TARGET]

assert set(train_df["match_id"]).isdisjoint(test_df["match_id"])
assert train_df["match_id"].nunique() == 102 and test_df["match_id"].nunique() == 26
assert len(train_df) == 204 and len(test_df) == 52  # team rows, not teams
assert y_train.sum() == 102 and y_test.sum() == 26  # one MOTM side per match

print("Train year counts:\n", train_df.groupby("year")["match_id"].nunique())
print("Test year counts:\n", test_df.groupby("year")["match_id"].nunique())

Train year counts:
 year
2018    51
2022    51
Name: match_id, dtype: int64
Test year counts:
 year
2018    13
2022    13
Name: match_id, dtype: int64


The test set is 20% of **matches** (26), not a random 20% of rows. With 26 test matches, **one match is worth about 3.85 percentage points** of accuracy (1 / 26), so small gaps between methods are noise. The test matches stay untouched until Phase 5.

## Phase 3: baseline

**Metric.** The model makes one pick per match, so I score **match accuracy**: the share of matches where the picked team supplied the Man of the Match. Within each match I pick the team with the higher score.

**Baseline.** A hand-written rule with no training. It's a yardstick, not a training stage: more goals → if tied, more shots on target → if tied, more possession → if still tied, a seeded coin flip. Its pick is the *predicted MOTM side*. Guessing randomly would get about 50%. The tiebreak chain belongs only to the baseline; the models learn their own weights.

In [6]:
baseline_train = motm.match_predictions(train_df, motm.baseline_scores(train_df))
train_draws = motm.drawn_match_ids(train_df)
on_draws = baseline_train[baseline_train["match_id"].isin(train_draws)]

print(f"Baseline on training matches: {baseline_train['correct'].mean():.3f}")
print(f"Baseline on drawn training matches: {on_draws['correct'].sum()}/{len(on_draws)}")
print("Random-choice expectation: 0.500")

Baseline on training matches: 0.902
Baseline on drawn training matches: 12/20
Random-choice expectation: 0.500


The baseline's test score is computed in Phase 5, on the same 26 matches as the models, draws included. Skipping draws would hide its hardest cases.

## Phase 4: training and tuning

Three models, from simple to more flexible, to test whether extra complexity helps:
- **Logistic regression:** gives each team row a probability and has readable weights. `C` sets flexibility (small = simpler). It is any positive number; only the *output* is between 0 and 1.
- **Decision tree:** a flowchart of yes/no questions. `max_depth` limits how many questions in a row, so it can't memorise 102 matches.
- **KNN:** the `k` most similar training rows vote. It uses distances, so features are scaled first.

Scaling happens per column (subtract the mean, divide by the spread), inside each pipeline, so it's fit on training folds only.

**Grouped 5-fold CV.** The 102 training matches are split into 5 groups. Each group takes one turn as the validation fold while the other 4 train, and the 5 scores are averaged for each setting. The best setting is then refit on all 102 matches. If two settings tie exactly, the simpler one wins. All tuning happens here; the test set is used once, at the end.

In [7]:
searches, best_by_set, fitted_by_set = {}, {}, {}

for set_name, features in FEATURE_SETS.items():
    searches[set_name] = motm.grouped_cv_search(train_df, features)
    best_by_set[set_name] = motm.best_settings(searches[set_name])
    fitted_by_set[set_name] = motm.fit_best(train_df, features, best_by_set[set_name])
    print(f"\nBest settings ({set_name}):")
    display(best_by_set[set_name][["model", "params", "cv_mean", "cv_std", "fold_scores"]])

# Choose each set's model from CV *before* looking at the test matches.
selected = {name: best.iloc[0]["model"] for name, best in best_by_set.items()}
print("Selected by grouped CV:", selected)


Best settings (with goals):


,model,params,cv_mean,cv_std,fold_scores
0,Logistic regression,{'model__C': 0.01},0.873810,0.115116,"[0.667, 0.952, 0.85, 0.9, 1.0]"
1,Decision tree,"{'model__max_depth': 2, 'model__min_samples_le...",0.832857,0.065990,"[0.762, 0.952, 0.8, 0.8, 0.85]"
2,KNN,{'model__n_neighbors': 11},0.774286,0.049611,"[0.714, 0.857, 0.75, 0.8, 0.75]"



Best settings (no goals):


,model,params,cv_mean,cv_std,fold_scores
0,Logistic regression,{'model__C': 10.0},0.688095,0.111433,"[0.476, 0.714, 0.75, 0.7, 0.8]"
1,KNN,{'model__n_neighbors': 3},0.674762,0.146830,"[0.619, 0.905, 0.7, 0.7, 0.45]"
2,Decision tree,"{'model__max_depth': 3, 'model__min_samples_le...",0.598095,0.159288,"[0.381, 0.81, 0.7, 0.45, 0.65]"


Selected by grouped CV: {'with goals': 'Logistic regression', 'no goals': 'Logistic regression'}


In [8]:
# How does each model's CV score change across its settings? (No-goals set.)
grid_view = searches["no goals"].assign(params=lambda t: t["params"].astype(str))
for model_name, rows in grid_view.groupby("model"):
    print(model_name)
    display(rows[["params", "cv_mean", "cv_std"]].sort_values("cv_mean", ascending=False).head(5))

Decision tree


,params,cv_mean,cv_std
10,"{'model__max_depth': 3, 'model__min_samples_le...",0.598095,0.159288
15,"{'model__max_depth': 5, 'model__min_samples_le...",0.596667,0.144150
13,"{'model__max_depth': 4, 'model__min_samples_le...",0.588095,0.183861
5,"{'model__max_depth': 2, 'model__min_samples_le...",0.586190,0.140338
6,"{'model__max_depth': 2, 'model__min_samples_le...",0.586190,0.140338


KNN


,params,cv_mean,cv_std
20,{'model__n_neighbors': 3},0.674762,0.146830
21,{'model__n_neighbors': 5},0.619048,0.159385
22,{'model__n_neighbors': 7},0.608095,0.107165
23,{'model__n_neighbors': 9},0.558571,0.095447
24,{'model__n_neighbors': 11},0.558571,0.070832


Logistic regression


,params,cv_mean,cv_std
3,{'model__C': 10.0},0.688095,0.111433
4,{'model__C': 100.0},0.688095,0.111433
2,{'model__C': 1.0},0.678571,0.129678
1,{'model__C': 0.1},0.658571,0.116444
0,{'model__C': 0.01},0.628571,0.090213


`cv_std` shows how much the 5 folds disagree; it isn't a confidence interval. Each fold has only about 20 matches, so one match moves a fold's score by about 5 points.

## Phase 5: evaluation

This is the first and only time the 26 test matches are used. Everything is scored on the same matches: the baseline, then each tuned model with and without goals.

In [9]:
predictions = {"Baseline rule": motm.match_predictions(test_df, motm.baseline_scores(test_df))}

for set_name, fitted_models in fitted_by_set.items():
    X_test = test_df[list(FEATURE_SETS[set_name])]
    for model_name, fitted in fitted_models.items():
        scores = motm.positive_scores(fitted, X_test)
        predictions[f"{model_name} ({set_name})"] = motm.match_predictions(test_df, scores)

results_table = pd.DataFrame(
    [motm.summarise(name, comparison) for name, comparison in predictions.items()]
)
display(results_table)
print("One test match =", round(100 / 26, 2), "percentage points")

,method,correct_matches,total_matches,match_accuracy
0,Baseline rule,21,26,0.808
1,Logistic regression (with goals),19,26,0.731
2,Decision tree (with goals),18,26,0.692
3,KNN (with goals),17,26,0.654
4,Logistic regression (no goals),18,26,0.692
5,KNN (no goals),11,26,0.423
6,Decision tree (no goals),13,26,0.500


One test match = 3.85 percentage points


In [10]:
# Year breakdown (13 matches each) and drawn matches, where goals can't decide.
test_draws = motm.drawn_match_ids(test_df)

def breakdown(comparison):
    by_year = comparison.groupby("year")["correct"].sum()
    drawn = comparison[comparison["match_id"].isin(test_draws)]["correct"]
    return {"2018 (/13)": int(by_year.get(2018, 0)), "2022 (/13)": int(by_year.get(2022, 0)),
            f"draws (/{len(test_draws)})": int(drawn.sum())}

display(pd.DataFrame({name: breakdown(c) for name, c in predictions.items()}).T)

,2018 (/13),2022 (/13),draws (/8)
Baseline rule,11,10,4
Logistic regression (with goals),11,8,3
Decision tree (with goals),10,8,2
KNN (with goals),8,9,3
Logistic regression (no goals),11,7,3
KNN (no goals),6,5,2
Decision tree (no goals),6,7,3


In [11]:
# Where did the main (no-goals, CV-selected) model go wrong?
main_name = f"{selected['no goals']} (no goals)"
main_comparison = predictions[main_name]
print("Main model:", main_name)
display(main_comparison.loc[~main_comparison["correct"]].sort_values("match_id"))

Main model: Logistic regression (no goals)


,match_id,predicted_team,selected_score,actual_team,year,correct
4,M-2018-21,Australia,0.614219,Denmark,2018,False
7,M-2018-29,Sweden,0.863793,Germany,2018,False
15,M-2022-06,Denmark,0.643910,Tunisia,2022,False
16,M-2022-14,South Korea,0.257570,Uruguay,2022,False
21,M-2022-42,Belgium,0.556190,Croatia,2022,False
22,M-2022-47,Brazil,0.800715,Cameroon,2022,False
23,M-2022-50,Australia,0.556769,Argentina,2022,False
25,M-2022-57,Brazil,0.950444,Croatia,2022,False


`selected_score` ranks the two teams within a match. It isn't a calibrated probability that a team wins the award: the two scores in a match don't have to add up to 1.

In [12]:
# Which stats did the main model lean on? Measured on training data only, and
# exploratory: correlated stats (shots, attempts, on-target) share importance.
from sklearn.inspection import permutation_importance

main_features = list(FEATURES_NO_GOALS)
main_model = fitted_by_set["no goals"][selected["no goals"]]
importance = permutation_importance(
    main_model, train_df[main_features], y_train,
    scoring="balanced_accuracy", n_repeats=20, random_state=42,
)
display(pd.DataFrame({
    "feature": main_features,
    "mean_importance": importance.importances_mean.round(3),
    "std_importance": importance.importances_std.round(3),
}).sort_values("mean_importance", ascending=False).head(10))

# Logistic regression weights (after scaling), for the no-goals set.
lr = fitted_by_set["no goals"]["Logistic regression"]
display(pd.Series(lr.named_steps["model"].coef_[0], index=main_features)
        .sort_values().round(3).to_frame("standardized_coefficient"))

,feature,mean_importance,std_importance
2,On-Target,0.126,0.029
0,Ball Possession %,0.054,0.019
4,Blocked,0.022,0.023
6,Offsides,0.021,0.014
1,Attempts,0.020,0.018
7,Free Kicks,0.015,0.018
5,Corners,0.013,0.019
13,Yellow Card,0.013,0.014
10,Passes,0.002,0.013
15,Red,0.002,0.003


,standardized_coefficient
Ball Possession %,-0.797
Blocked,-0.447
Red,-0.362
Offsides,-0.346
Yellow Card,-0.322
Attempts,-0.322
Distance Covered (Kms),-0.116
Yellow & Red,-0.035
Fouls Committed,0.042
Off-Target,0.101


## Write-up

*Write this part in your own words, using the numbers above.*

- **Method:** features (with and without goals), the grouped 102/26 split, the baseline, the three models and grouped 5-fold CV.
- **Results:** baseline [x]/26; with goals: LR [x], tree [x], KNN [x]; no goals: LR [x], tree [x], KNN [x]. Main model [x]/26, with [x]/13 in 2018 and [x]/13 in 2022.
- **Interpretation:** how much do the other stats add on top of goals? Does the no-goals model beat the baseline on drawn matches? If it doesn't beat the baseline overall, say so; that's an honest finding.
- **Limits:** 26 test matches (1 match ≈ 3.85 points), different 2018/2022 sources, no per-90 scaling for extra time, and the model predicts the team, not the player.